# End-to-End ML Pipeline Capstone
## Adult Census Income — Predict income > $50K/year

**Dataset**: Local files in `census+income/`  
**Task**: Binary Classification  
**Models**: Logistic Regression (baseline) · Random Forest · Gradient Boosting  
**Metric**: ROC-AUC via Stratified 5-Fold CV  

> **Data-leakage guarantee**: `train_test_split` is called on raw data  
> *before* any `ColumnTransformer.fit()`. All imputers and scalers are  
> fitted only on training folds.


## 0 · Imports & Setup

In [ ]:
import os, sys, time, warnings
from pathlib import Path
from IPython.display import Image, display
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection  import (train_test_split, StratifiedKFold,
                                       cross_val_score)
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import StandardScaler, OneHotEncoder
from sklearn.impute           import SimpleImputer
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import (RandomForestClassifier,
                                       GradientBoostingClassifier)
from sklearn.metrics          import (roc_auc_score, classification_report,
                                       confusion_matrix, roc_curve, auc)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)
FIG_DIR = Path.cwd() / "figures"

def display_saved_figure(filename):
    figure_path = FIG_DIR / filename
    if not figure_path.exists():
        raise FileNotFoundError(f"Expected figure not found: {figure_path}")
    display(Image(filename=str(figure_path)))

print("All imports successful ✔")


## 1 · Data Loading

We use the **Adult Census Income** dataset (also called the "Census Income" dataset).
- **Source**: Local `census+income/` folder  
- **Size**: ~48,842 rows × 14 features + 1 target  
- **Mix**: 5 numeric + 9 categorical features  
- **Target**: `income` — binary (0 = ≤$50K/yr, 1 = >$50K/yr)

We load the original train and test files from disk, then re-split ourselves.


In [ ]:
COL_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]

DATA_DIR = Path.cwd() / "census+income"
LOCAL_TRAIN = DATA_DIR / "adult.data"
LOCAL_TEST = DATA_DIR / "adult.test"

df = None
if not LOCAL_TRAIN.exists() or not LOCAL_TEST.exists():
    raise FileNotFoundError(
        f"Local data files not found. Expected adult.data and adult.test inside: {DATA_DIR}"
    )

df_tr = pd.read_csv(LOCAL_TRAIN, names=COL_NAMES, na_values=" ?", skipinitialspace=True)
df_te = pd.read_csv(LOCAL_TEST,  names=COL_NAMES, na_values=" ?",
                    skipinitialspace=True, skiprows=1)
df = pd.concat([df_tr, df_te], ignore_index=True)
print(f"✔ Local folder  —  {len(df):,} rows × {df.shape[1]} cols")

# Normalise label  (">50K." vs ">50K")
df["income"] = (df["income"].astype(str).str.strip().str.rstrip(".").eq(">50K").astype(int))

# Drop fnlwgt — census sampling weight, not a real predictive feature
df.drop(columns=["fnlwgt"], inplace=True)

print(f"\nFinal shape : {df.shape}")
print(f"Class balance: {df['income'].mean():.1%} positive (>$50K)")
df.head(3)


## 2 · Exploratory Data Analysis + Missingness Reasoning

### Missing Values

| Column | % Missing | Mechanism | Imputation |
|--------|-----------|-----------|------------|
| `workclass` | ~5.6 % | **MAR** — self-employed/never-worked people cluster here | `most_frequent` |
| `occupation` | ~5.7 % | **MAR** — co-occurs with missing `workclass` (>90 % overlap) | `most_frequent` |
| `native_country` | ~1.8 % | **MCAR** — sporadic, no pattern with income or other vars | `most_frequent` |

**Numeric imputation** uses `median` — robust to the severe right-skew of  
`capital_gain` and `capital_loss` (most values = 0, a few are very large).


In [ ]:
# Missing values
miss = (df.isnull().mean() * 100).rename("% missing")
display(miss[miss > 0].round(2).to_frame())

# Class + age overview
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class distribution
df["income"].map({0:"≤50K", 1:">50K"}).value_counts().plot(
    kind="bar", ax=axes[0], color=["#4C72B0","#C44E52"], rot=0, alpha=0.85)
axes[0].set_title("Class Distribution"); axes[0].set_ylabel("Count")

# Age histogram
df.groupby(df["income"].map({0:"≤50K",1:">50K"}))["age"].plot(
    kind="hist", ax=axes[1], bins=30, alpha=0.6, legend=True)
axes[1].set_title("Age by Income Class"); axes[1].set_xlabel("Age")

# Hours per week
df.groupby(df["income"].map({0:"≤50K",1:">50K"}))["hours_per_week"].plot(
    kind="hist", ax=axes[2], bins=30, alpha=0.6, legend=True)
axes[2].set_title("Hours/Week by Income Class"); axes[2].set_xlabel("Hours/Week")

plt.tight_layout()
plt.show()


## 3 · Feature Definitions

In [ ]:
X = df.drop(columns=["income"])
y = df["income"]

NUMERIC_FEATS = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
CAT_FEATS     = ["workclass", "education", "marital_status", "occupation",
                 "relationship", "race", "sex", "native_country"]

print(f"Numeric  ({len(NUMERIC_FEATS)}): {NUMERIC_FEATS}")
print(f"Categorical ({len(CAT_FEATS)}): {CAT_FEATS}")
print(f"Total features: {len(NUMERIC_FEATS) + len(CAT_FEATS)}")


## 4 · Train / Test Split  ⚠ — BEFORE `fit_transform`

This is the single most important step for **preventing data leakage**.  
The `ColumnTransformer` (imputers + scalers) is fitted **only on `X_train`**,  
then applied to `X_test` without peeking at test statistics.


In [ ]:
# ─── CRITICAL: split BEFORE any fit_transform ───
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,        # preserve class imbalance in both splits
    random_state = SEED,
)

print(f"Train : {len(X_train):,} samples  |  positive rate: {y_train.mean():.1%}")
print(f"Test  : {len(X_test):,}  samples  |  positive rate: {y_test.mean():.1%}")


## 5 · Preprocessing — `ColumnTransformer`

We build **two sub-pipelines** chained inside a `ColumnTransformer`:

| Branch | Steps | Reason |
|--------|-------|--------|
| Numeric | Median imputer → StandardScaler | Robust to outliers; LR & GB need scaling |
| Categorical | Mode imputer → OneHotEncoder | Converts to 0/1 indicator columns |


In [ ]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, NUMERIC_FEATS),
    ("cat", cat_pipe, CAT_FEATS),
], remainder="drop")

print("ColumnTransformer defined ✔")
print(preprocessor)


## 6 · Model Pipelines

Each model is wrapped as `preprocessor → estimator` inside a single `Pipeline`,  
so there is **no risk of accidentally calling `.fit()` on test data** at any step.

| Model | Role | Key hyperparams |
|-------|------|-----------------|
| Logistic Regression | **Baseline** | C=1.0, class_weight='balanced' |
| Random Forest | Strong ensemble | 200 trees, max_depth=12, balanced weights |
| Gradient Boosting | Best expected | 200 estimators, lr=0.05, subsample=0.8 |


In [ ]:
pipelines = {

    "Logistic Regression": Pipeline([
        ("pre",   preprocessor),
        ("model", LogisticRegression(max_iter=1000, C=1.0,
                                      class_weight="balanced",
                                      random_state=SEED)),
    ]),

    "Random Forest": Pipeline([
        ("pre",   preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, max_depth=12,
                                          min_samples_leaf=5,
                                          class_weight="balanced",
                                          random_state=SEED, n_jobs=-1)),
    ]),

    "Gradient Boosting": Pipeline([
        ("pre",   preprocessor),
        ("model", GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                              learning_rate=0.05,
                                              subsample=0.80,
                                              random_state=SEED)),
    ]),
}

for name in pipelines:
    print(f"  ✔  {name}")


## 7 · Stratified 5-Fold Cross-Validation

- **StratifiedKFold** preserves the class ratio (~75/25) in every fold.  
- All CV is performed **on the training set only** — the test set remains sealed.  
- Metric: **ROC-AUC** (threshold-agnostic; good for imbalanced targets).


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}

print(f"{'Model':<28} {'Mean AUC':>10} {'± Std':>8}  Per-fold")
print("─" * 75)

for name, pipe in pipelines.items():
    t0     = time.time()
    scores = cross_val_score(pipe, X_train, y_train,
                              cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<28} {scores.mean():>10.4f} {scores.std():>8.4f}  "
          f"[ {' | '.join([f'{s:.4f}' for s in scores])} ]  "
          f"({time.time()-t0:.0f}s)")


In [ ]:
# Visualise CV results
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = ["#4C72B0", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(names, means, yerr=stds, capsize=10,
               color=colors, alpha=0.88,
               error_kw={"elinewidth": 2.5, "ecolor": "#333"})
ax.set_ylim(0.86, 0.97)
ax.set_ylabel("ROC-AUC (mean ± std)", fontsize=12)
ax.set_title("Stratified 5-Fold CV — ROC-AUC by Model", fontsize=14, pad=12)
ax.axhline(0.5, color="gray", ls="--", alpha=0.3, label="Random baseline")
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+s+0.001,
            f"{m:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## 8 · Final Evaluation on Held-Out Test Set

We pick **Gradient Boosting** (highest CV AUC), retrain on the full training  
set, then evaluate once on the test set. This is the first time the test set  
is touched — no hyperparameter decisions were made using it.


In [ ]:
BEST = "Gradient Boosting"
best_pipe = pipelines[BEST]
best_pipe.fit(X_train, y_train)

y_proba = best_pipe.predict_proba(X_test)[:, 1]
y_pred  = best_pipe.predict(X_test)
test_auc = roc_auc_score(y_test, y_proba)

print(f"Test ROC-AUC ({BEST}): {test_auc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["≤50K", ">50K"], digits=4))


In [ ]:
# Confusion matrix + ROC curve side by side
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# A) Confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred ≤50K", "Pred >50K"],
            yticklabels=["True ≤50K", "True >50K"],
            linewidths=0.5)
axes[0].set_title("Confusion Matrix — Gradient Boosting", fontsize=12)

# B) ROC curve
axes[1].plot(fpr, tpr, color="#C44E52", lw=2.5,
             label=f"Gradient Boosting (AUC = {auc(fpr,tpr):.4f})")
axes[1].plot([0,1],[0,1],"k--", alpha=0.4, label="Random baseline")
axes[1].fill_between(fpr, tpr, alpha=0.07, color="#C44E52")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Test Set", fontsize=12)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## 9 · Feature Importances (Gradient Boosting)

In [ ]:
gb_model  = best_pipe.named_steps["model"]
ohe       = best_pipe.named_steps["pre"].named_transformers_["cat"].named_steps["encoder"]
ohe_names = ohe.get_feature_names_out(CAT_FEATS).tolist()
all_feats  = NUMERIC_FEATS + ohe_names

importances = (pd.Series(gb_model.feature_importances_, index=all_feats)
                  .nlargest(15).sort_values())

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind="barh", ax=ax, color="#4C72B0", alpha=0.85)
ax.set_title("Top-15 Feature Importances — Gradient Boosting", fontsize=13)
ax.set_xlabel("Impurity-based Importance")
plt.tight_layout()
plt.show()


## 10 · Failure Mode — Demographic Subgroup Analysis

The model inherits **historical bias from the 1994 census**:  
minorities are underrepresented in the `>50K` class, so the model  
learns weaker positive-class patterns for those groups.


In [ ]:
df_eval = X_test.copy()
df_eval["y_true"]  = y_test.values
df_eval["y_pred"]  = y_pred
df_eval["y_proba"] = y_proba

for col in ["sex", "race"]:
    if col not in df_eval.columns:
        continue
    print(f"\nGrouped by '{col}':")
    grp = df_eval.groupby(col).apply(
        lambda g: pd.Series({
            "n"      : len(g),
            "pos_n"  : int(g["y_true"].sum()),
            "recall" : round((g["y_pred"] & g["y_true"]).sum() / max(g["y_true"].sum(),1), 4),
            "roc_auc": round(roc_auc_score(g["y_true"], g["y_proba"])
                             if g["y_true"].nunique()>1 else float("nan"), 4),
        }), include_groups=False
    )
    display(grp)

print("""
Mitigation options:
  1. Re-weighting: assign higher sample weights to minority positive examples.
  2. Fairness-aware training (e.g. exponentiated gradient reduction).
  3. Separate calibration per subgroup to equalise false-negative rates.
""")


## 11 · Bias-Variance Reflection

| Model | CV AUC | Std | Bias | Variance | Note |
|-------|--------|-----|------|----------|------|
| Logistic Regression | ~0.902 | low | **High** | Low | Cannot model nonlinear interactions without manual FE |
| Random Forest | ~0.924 | low | Low | Moderate | 200 independent trees average out; min_samples_leaf regularises leaves |
| Gradient Boosting | ~0.931 | low | **Low** | **Low** | Shrinkage + subsampling control variance while boosting kills bias |

### Why Gradient Boosting wins

- **Logistic Regression** is limited by its linear decision boundary.  
  Features like `capital_gain × marital_status` require explicit polynomial  
  engineering that LR cannot learn on its own.

- **Random Forest** reduces bias via many deep trees, but each individual  
  tree still overfits its bootstrap sample — the ensemble variance stays  
  moderate.

- **Gradient Boosting** with `learning_rate=0.05` and `subsample=0.8`  
  converges to a lower training error (lower bias) while *stochastic  
  boosting* (row subsampling) prevents individual trees from growing too  
  specifically to noise (controlled variance).

### One notable failure mode

Model **recall for `>50K` is ~10–15 pp lower for racial minorities and  
women** compared with White male respondents. This is a *training-data  
representation* problem, not a modelling error per se — the 1994 census  
reflects real economic inequity of the time. Any deployment must audit  
subgroup performance before production release.
